# maskx + Equinox + Optax demo

This notebook builds a small Equinox MLP, uses `maskx` to select only the final layer parameters, and trains only those parameters with Optax.

The point of the example is to show that `maskx` gives you a path-based way to decide which leaves are trainable.

In [10]:
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr
import maskx
import optax

In [21]:
key = jr.PRNGKey(0)
model = eqx.nn.MLP(in_size=1, out_size=1, width_size=16, depth=4, key=key)

xs = jnp.linspace(-1.0, 1.0, 128).reshape(-1, 1)
ys = jnp.sin(3.0 * xs)

trainable = maskx.select(model, target=r"layers/2/.*", leaf_type=jax.Array)

print('Selected parameter paths:')
for path in trainable.paths():
    print(' -', path)
print('Selected leaf count:', trainable.count())

Selected parameter paths:
 - layers/2/weight
 - layers/2/bias
Selected leaf count: 2


`optax.multi_transform` expects a label tree. We derive that label tree from the boolean mask returned by `maskx`.

Only the selected leaves get the Adam transform; everything else gets `set_to_zero()`, which freezes those parameters.

In [22]:
labels = jax.tree_util.tree_map(
    lambda selected: 'train' if selected else 'freeze',
    trainable.tree,
)

optimizer = optax.multi_transform(
    {
        'train': optax.adam(1e-2),
        'freeze': optax.set_to_zero(),
    },
    lambda params: labels,
)

opt_state = optimizer.init(model)

@eqx.filter_value_and_grad
def loss_fn(model, x, y):
    preds = jax.vmap(model)(x)
    return jnp.mean((preds - y) ** 2)

@eqx.filter_jit
def step(model, opt_state, x, y):
    loss, grads = loss_fn(model, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

In [ ]:
initial_first_layer = model.layers[0].weight.copy()
initial_last_layer = model.layers[-1].weight.copy()
initial_loss, _ = loss_fn(model, xs, ys)

loss_history = [float(initial_loss)]
for _ in range(10_000):
    model, opt_state, loss = step(model, opt_state, xs, ys)
    loss_history.append(float(loss))

first_layer_frozen = bool(
    jnp.allclose(initial_first_layer, model.layers[0].weight)
 )
last_layer_changed = not bool(
    jnp.allclose(initial_last_layer, model.layers[-1].weight)
 )

print(f'Initial loss: {loss_history[0]:.6f}')
print(f'Final loss:   {loss_history[-1]:.6f}')
print('First layer frozen:', first_layer_frozen)
print('Last layer changed:', last_layer_changed)

Initial loss: 0.521103
Final loss:   0.000050
First layer frozen: True
Last layer changed: False


In [ ]:
sample_x = jnp.array([[-1.0], [-0.5], [0.0], [0.5], [1.0]])
sample_y = jnp.sin(3.0 * sample_x)
sample_pred = jax.vmap(model)(sample_x)

for x_value, target, pred in zip(
    sample_x[:, 0], sample_y[:, 0], sample_pred[:, 0]
 ):
    row = (
        f'x={float(x_value): .1f}  '
        f'target={float(target): .4f}  '
        f'pred={float(pred): .4f}'
    )
    print(row)

x=-1.0  target=-0.1411  pred=-0.1603
x=-0.5  target=-0.9975  pred=-1.0005
x= 0.0  target= 0.0000  pred= 0.0066
x= 0.5  target= 0.9975  pred= 0.9942
x= 1.0  target= 0.1411  pred= 0.1485
